In [10]:
from astropy.table import Table
import numpy as np
from tensorflow.keras.utils import to_categorical

# Sort through the data to only get 'real' data, no noise. For example the S/N must be larger than 3 
data = Table.read('data/extra_data_50000_z0.3-0.8_classified.csv',format='ascii.csv',header_start=0,data_start=1)

# ensure no NaN
valid_sigma_o3_err = ((data['Sigma_OIII_4363_Err1']+data['Sigma_OIII_4958_Err']) > 0) & np.isfinite((data['Sigma_OIII_4363_Err1']+data['Sigma_OIII_4958_Err']))  # valid error values
valid_sigma_o3 = np.isfinite((data['Sigma_OIII_43631'] + data['Sigma_OIII_4958']))  # valid sigma_o3 values
valid_data = valid_sigma_o3 & valid_sigma_o3_err  # only keep valid data points
print(data.colnames)

['specObjID', 'z', 'sigmaStars', 'sigmaStarsErr', 'redshift', 'bpt', 'Flux_OII_3726', 'Flux_OII_3726_Err', 'Flux_OII_3728', 'Flux_OIII_4363', 'Flux_OIII_4958', 'Flux_OIII_5006', 'Flux_NII_6547', 'Flux_SII_6716', 'Flux_OII_3728_Err', 'Flux_OIII_4363_Err', 'Flux_OIII_4958_Err', 'Flux_OIII_5006_Err', 'Flux_NII_6547_Err', 'Flux_SII_6716_Err', 'Sigma_OIII_4363', 'Sigma_OIII_4363_Err', 'Sigma_OIII_43631', 'Sigma_OIII_4363_Err1', 'Sigma_OIII_4958', 'Sigma_OIII_4958_Err', 'Flux_Hb_4861', 'Flux_Hb_4861_Err', 'Flux_Ha_6562', 'Flux_Ha_6562_Err', 'u', 'g', 'r', 'i', 'z1']


In [13]:
# Filter out stuff
# I was tired of the dividing by zero errors so first just filter out the zeros
valid_errors = (data['Flux_OIII_5006_Err'] > 0) & (data['Flux_OII_3726_Err'] > 0) & (data['Flux_Hb_4861_Err'] > 0) & (data['Flux_Ha_6562_Err'] > 0) & (data['Flux_SII_6716_Err'] > 0)

# In the paper they use [OIII] 5007, so I guess the 5006 here is what we should use!
ind1=np.where(np.array(data['Flux_OIII_5006']/data['Flux_OIII_5006_Err']) >3)
ind2=np.where(np.array((data['Flux_OII_3726']+data['Flux_OII_3728'])/np.sqrt(data['Flux_OII_3726_Err']**2+data['Flux_OII_3728_Err']**2)) >3)
ind3=np.where(np.array(data['Flux_Hb_4861']/data['Flux_Hb_4861_Err']) >3)
ind4=np.where(np.array(data['Flux_Ha_6562']/data['Flux_Ha_6562_Err']) >3)
ind5=np.where(np.array(data['Flux_SII_6716']/data['Flux_SII_6716_Err']) >3)

# I squared the errors as they did for [0II] since we have sigma_03 at 2 lines
ind6 = np.where(valid_data & ((data['Sigma_OIII_4363']+data['Sigma_OIII_4958']) / (data['Sigma_OIII_4363_Err1']**2+data['Sigma_OIII_4958_Err']**2) > 3)) 
ind7=np.where(np.array((data['Sigma_OIII_4363']+data['Sigma_OIII_4958']) / np.sqrt(data['Sigma_OIII_4363_Err1']**2+data['Sigma_OIII_4958_Err']**2) >3))
ind8=np.where(np.array(data['sigmaStars'])>0.)

# I have the magnitudes already, so I don't filter for flux/ calculate magnitudes

# Sort through these indices
ind=np.array(list(set(ind1[0]) & set(ind2[0]) & set(ind3[0]) & set(ind4[0]) & set(ind5[0]) & set(ind6[0]) & set(ind7[0]) & set(ind8[0])))# & set(ind9[0])& set(ind10[0]) ))
n_source=len(ind)
print(n_source)

154


C:\Users\cstij\AppData\Local\Temp\ipykernel_72632\666033025.py:6: RuntimeWarning: divide by zero encountered in divide
  ind1=np.where(np.array(data['Flux_OIII_5006']/data['Flux_OIII_5006_Err']) >3)
C:\Users\cstij\AppData\Local\Temp\ipykernel_72632\666033025.py:6: RuntimeWarning: invalid value encountered in divide
  ind1=np.where(np.array(data['Flux_OIII_5006']/data['Flux_OIII_5006_Err']) >3)
C:\Users\cstij\AppData\Local\Temp\ipykernel_72632\666033025.py:7: RuntimeWarning: divide by zero encountered in divide
  ind2=np.where(np.array((data['Flux_OII_3726']+data['Flux_OII_3728'])/np.sqrt(data['Flux_OII_3726_Err']**2+data['Flux_OII_3728_Err']**2)) >3)
C:\Users\cstij\AppData\Local\Temp\ipykernel_72632\666033025.py:7: RuntimeWarning: invalid value encountered in divide
  ind2=np.where(np.array((data['Flux_OII_3726']+data['Flux_OII_3728'])/np.sqrt(data['Flux_OII_3726_Err']**2+data['Flux_OII_3728_Err']**2)) >3)
C:\Users\cstij\AppData\Local\Temp\ipykernel_72632\666033025.py:8: RuntimeWarning

In [14]:
print(data.colnames)

['specObjID', 'z', 'sigmaStars', 'sigmaStarsErr', 'redshift', 'bpt', 'Flux_OII_3726', 'Flux_OII_3726_Err', 'Flux_OII_3728', 'Flux_OIII_4363', 'Flux_OIII_4958', 'Flux_OIII_5006', 'Flux_NII_6547', 'Flux_SII_6716', 'Flux_OII_3728_Err', 'Flux_OIII_4363_Err', 'Flux_OIII_4958_Err', 'Flux_OIII_5006_Err', 'Flux_NII_6547_Err', 'Flux_SII_6716_Err', 'Sigma_OIII_4363', 'Sigma_OIII_4363_Err', 'Sigma_OIII_43631', 'Sigma_OIII_4363_Err1', 'Sigma_OIII_4958', 'Sigma_OIII_4958_Err', 'Flux_Hb_4861', 'Flux_Hb_4861_Err', 'Flux_Ha_6562', 'Flux_Ha_6562_Err', 'u', 'g', 'r', 'i', 'z1']


In [24]:
type_arr=np.zeros(len(ind))
type_arr=type_arr-999

# Give the labels numbers:
type_arr = data['bpt'][ind]
# print(type_arr)
type_arr[type_arr == 'Star Forming'] = 1
type_arr[type_arr == 'Composite'] = 2
type_arr[type_arr == 'Seyfert'] = 3
type_arr[type_arr == 'LINER'] = 4

# Check the if we have any of the unclear class
for t in type_arr:
    if t == 'Seyfert/LINER':
        print('Oh no, we have a Seyfert/LINER in the data!')
        break

# Get/define the 8 input features
z=np.array(data['z'][ind])
O2_index=np.log10((data['Flux_OII_3726'][ind]+data['Flux_OII_3728'][ind])/data['Flux_Hb_4861'][ind])
O3_index=np.log10(data['Flux_OIII_5006'][ind]/data['Flux_Hb_4861'][ind])
N2_index=np.log10(data['Flux_NII_6547'][ind]/data['Flux_Ha_6562_Err'][ind])
S2_index=np.log10(data['Flux_SII_6716'][ind]/data['Flux_Ha_6562_Err'][ind])
sigma_o3=np.log10((data['Sigma_OIII_4363'][ind]+data['Sigma_OIII_4958'][ind]))
sigma_star=np.log10(data['sigmaStars'][ind])
u_g=data['u'][ind]-data['g'][ind]
g_r=data['g'][ind]-data['r'][ind]
r_i=data['r'][ind]-data['i'][ind]
i_z=data['i'][ind]-data['z1'][ind]


# Creating a dictionary with all X values
features = {
        'O2_index': np.log10((data['Flux_OII_3726'][ind] + data['Flux_OII_3728'][ind]) / data['Flux_Hb_4861'][ind]),
        'O3_index': np.log10(data['Flux_OIII_5006'][ind] / data['Flux_Hb_4861'][ind]),
        'sigma_o3': np.log10(data['Sigma_OIII_4363'][ind]),
        'sigma_star': np.log10(data['sigmaStars'][ind]),
        'u_g': data['u'][ind] - data['g'][ind],
        'g_r': data['g'][ind] - data['r'][ind],
        'r_i': data['r'][ind] - data['i'][ind],
        'i_z': data['i'][ind] - data['z1'][ind],
    }


# Create the final arrays from the dictionaries
X= np.column_stack([features['O2_index'], features['O3_index'], features['sigma_o3'],
                           features['sigma_star'], features['u_g'], features['g_r'], 
                           features['r_i'], features['i_z']])


C:\Users\cstij\AppData\Local\Temp\ipykernel_72632\3985308439.py:22: RuntimeWarning: divide by zero encountered in log10
  N2_index=np.log10(data['Flux_NII_6547'][ind]/data['Flux_Ha_6562_Err'][ind])


In [32]:
valid_indices = type_arr!= -999
X= X[valid_indices]
type_arr = type_arr[valid_indices] 
type_arr = np.array(type_arr, dtype=int)
# One hot encoding of the labels
y = to_categorical(type_arr - 1, num_classes=4)

np.save('data/X_extra_data.npy', X)
np.save('data/labels_extra_data.npy',y)